# LIBERO Simulation Benchmark — Dependency Setup

Installs everything needed to run `benchmark_libero.py` against a trained `TransformerFlowMatchingPolicy`.

**Target**: Google Colab (Linux) with GPU, headless rendering via Xvfb + EGL.

**Order matters**:
1. Pin `numpy<2.0` first (robosuite 1.4.x + numba require it).
2. Install system libs for headless rendering.
3. Clone the `lerobot-piper` repo so `benchmark_libero.py` and the policy code are on disk.
4. Install LIBERO from source with `--no-deps` so it does not downgrade numpy to a broken 1.22.
5. Install the remaining simulation and policy dependencies.
6. Restart the kernel once, then verify imports.
7. (Optional) Smoke-test rendering.
8. Run the full benchmark.

## 1. Pin NumPy < 2.0

robosuite 1.4.1 and numba pin to `numpy<2`. Doing this first prevents later installs from picking up an incompatible 2.x wheel.

In [ ]:
!pip install "numpy<2.0"

## 2. System Dependencies (Headless Rendering)

Required when running without a physical display (Colab, remote servers). `MUJOCO_GL=egl` gives GPU-accelerated offscreen rendering.

In [ ]:
!apt-get install -y xvfb libgl1-mesa-glx libegl1-mesa libgles2-mesa-dev 2>/dev/null

## 3. Clone the `lerobot-piper` Repo

`benchmark_libero.py` lives in this repo and imports `models.transformer_flow_matching` from `src/`. Cloning here makes both the script and the policy class available.

In [ ]:
import os
if not os.path.exists('lerobot-piper'):
    !git clone https://github.com/mayuanyang/lerobot-piper.git

!ls lerobot-piper/src/benchmark_libero.py

## 4. Install LIBERO (from source, no deps)

`--no-deps` is critical — LIBERO's `setup.py` pins `numpy==1.22.4` which is broken on modern Python. We install its real dependencies in the next cell.

In [ ]:
import os
if not os.path.exists('LIBERO'):
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git

!pip install -e ./LIBERO --no-deps
!pip install hydra-core==1.2.0

## 5. Simulation Dependencies

Versions match what `benchmark_libero.py` is tested against. `robosuite==1.4.1` is required — newer 1.5.x has API changes the LIBERO env loaders cannot handle, and `benchmark_libero.py` applies compatibility monkeypatches specifically for 1.4.1.

In [ ]:
!pip install bddl robosuite==1.4.1 mujoco
!pip install pyvirtualdisplay

## 6. Policy and Dataset Dependencies

`lerobot==0.4.0` provides `LeRobotDatasetMetadata` for normalization stats. `transformers` is needed by `TransformerFlowMatchingPolicy` (SmolVLM vision backbone).

**Why the final `numpy<2` re-pin?** `lerobot==0.4.0`'s dependency resolver often pulls numpy 2.x in as a transitive upgrade, which then breaks robosuite/numba at import time. Force-reinstalling numpy<2 as the last step before the kernel restart fixes it.

In [ ]:
!pip install lerobot==0.4.0
!pip install transformers
!pip install decord opencv-python num2words

# A transitive dep above (typically lerobot's resolver) often re-installs numpy 2.x.
# Force it back to <2 before the kernel restart so robosuite/numba don't blow up.
!pip install --force-reinstall "numpy<2.0"

## 7. Restart Kernel

After the numpy downgrade and the new C-extension installs (mujoco, robosuite), a one-time kernel restart is required so already-imported modules pick up the correct binaries.

**Run the cell below — it will restart the kernel automatically. Then re-open the notebook and continue from the verification cell.**

In [ ]:
import os
os.kill(os.getpid(), 9)

## 8. Verify Installation

After restart, run this cell. It mirrors the imports `benchmark_libero.py` performs, applies the same robosuite monkeypatch, and confirms the policy class is importable from the cloned repo.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import pandas as pd
import huggingface_hub

assert int(np.__version__.split('.')[0]) < 2, f"numpy {np.__version__} — must be <2.0"

# Apply the same robosuite 1.4.1 compatibility patches benchmark_libero.py uses
import json
import robosuite.controllers as controllers
import robosuite.robots as robots
from robosuite.robots.robot import Robot

def _load_controller_config(custom_fpath=None, default_controller=None):
    if custom_fpath is not None:
        with open(custom_fpath) as f:
            return json.load(f)
    return {"type": default_controller} if default_controller else {}

if not hasattr(controllers, "load_controller_config"):
    setattr(controllers, "load_controller_config", _load_controller_config)
if not hasattr(robots, "SingleArm"):
    setattr(robots, "SingleArm", Robot)

from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
import mujoco
import pyvirtualdisplay

# Make the cloned lerobot-piper/src importable so we can load the policy class
repo_src = Path("lerobot-piper/src").resolve()
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from models.transformer_flow_matching.transformer_flow_matching_policy import (
    TransformerFlowMatchingPolicy,
)

print(f"numpy        : {np.__version__}")
print(f"torch        : {torch.__version__}  (cuda={torch.cuda.is_available()})")
print(f"mujoco       : {mujoco.__version__}")
print(f"LIBERO suites: {list(benchmark.get_benchmark_dict().keys())}")
print(f"Policy class : {TransformerFlowMatchingPolicy.__name__} importable from {repo_src}")
print("\nAll dependencies for benchmark_libero.py are installed.")

## 9. (Optional) Smoke-Test Headless Rendering

Boots a virtual display, builds one LIBERO env, and resets it. Confirms the EGL + Xvfb pipeline works before launching a long benchmark run.

In [ ]:
import os
from pyvirtualdisplay import Display

display = Display(visible=0, size=(1400, 900))
display.start()
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("DISPLAY", ":99")

task_suite = benchmark.get_benchmark_dict()["libero_spatial"]()
env = OffScreenRenderEnv(
    bddl_file_name=task_suite.get_task_bddl_file_path(0),
    camera_heights=256,
    camera_widths=256,
)
obs = env.reset()
env.close()

print("Headless render OK. Observation keys:")
for k, v in obs.items():
    arr = np.array(v)
    print(f"  {k:40s}  shape={arr.shape}  dtype={arr.dtype}")

## 10. Run the Full Benchmark

Runs `benchmark_libero.py` across all four LIBERO suites (`libero_spatial`, `libero_object`, `libero_goal`, `libero_long`).

**Paper-standard cost**: 50 rollouts × 10 tasks × 4 suites = 2000 rollouts. Expect several hours on a single GPU.

For a fast sanity check first, drop `NUM_ROLLOUTS` to 5 and `MAX_STEPS` to 150. Results are written to `OUTPUT_JSON` so you can re-load them without re-running.

In [ ]:
CHECKPOINT     = "ISdept/fm64-libero"   # HF repo ID or local checkpoint dir
NUM_ROLLOUTS   = 50                      # paper standard
MAX_STEPS      = 300
N_ACTION_STEPS = 8
IMAGE_SIZE     = 256
OUTPUT_JSON    = "libero_benchmark_results.json"
LOG_FILE       = "benchmark.log"

# `python -u` disables stdout buffering so prints stream live in Colab.
# `2>&1 | tee` mirrors output to both the cell and a log file (Colab truncates
# very long cell outputs, so the log file is the durable copy).
!python -u lerobot-piper/src/benchmark_libero.py \
    --checkpoint {CHECKPOINT} \
    --suites libero_spatial libero_object libero_goal libero_long \
    --num_rollouts {NUM_ROLLOUTS} \
    --max_steps {MAX_STEPS} \
    --n_action_steps {N_ACTION_STEPS} \
    --image_size {IMAGE_SIZE} \
    --output_json {OUTPUT_JSON} \
    --headless \
    2>&1 | tee {LOG_FILE}

In [ ]:
# Reload the saved JSON results into a DataFrame
import json
import pandas as pd

with open(OUTPUT_JSON) as f:
    all_results = json.load(f)

rows = []
for suite, data in all_results.items():
    for task_name, td in data["tasks"].items():
        rows.append({
            "Suite":    suite,
            "Task":     task_name,
            "Success":  f"{td['successes']}/{td['total']}",
            "Rate (%)": round(td["rate"], 1),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print("\nSuite averages:")
for suite, data in all_results.items():
    print(f"  {suite:20s}  {data['suite_avg']:5.1f}%")

## 11. Inspect Executed Actions

Two ways to see what the policy actually predicted during rollouts.

**(a) Lightweight per-step log across the full benchmark.** Re-run the benchmark cell above with `--actions_log libero_actions.npz` appended. Overhead is negligible — every executed action plus the matching state is appended to one `.npz`. Load and plot below.

**(b) Deep-inspect one episode.** Runs a single rollout with full per-step recording (state, action, both camera frames). Cheap; use it to drill into a specific task or failure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ACTIONS_LOG = "libero_actions.npz"   # set --actions_log to this in the benchmark cell above

data = np.load(ACTIONS_LOG, allow_pickle=True)
suites         = data["suites"].astype(str)
task_ids       = data["task_ids"]
rollouts       = data["rollouts"]
steps          = data["steps"]
states         = data["states"]
actions_norm   = data["actions_norm"]
actions_unnorm = data["actions_unnorm"]

# Count distinct rollouts
ids = np.stack([suites, task_ids.astype(str), rollouts.astype(str)]).T
n_rollouts = len(np.unique(ids, axis=0))
print(f"Logged {len(steps)} steps across {n_rollouts} rollouts "
      f"({len(np.unique(suites))} suites)")
print(f"action_unnorm: shape={actions_unnorm.shape}")
print(f"  per-dim min: {actions_unnorm.min(0)}")
print(f"  per-dim max: {actions_unnorm.max(0)}")

# Z-stats of normalized actions: if the policy stays in-distribution these are ~N(0,1).
# A dim with |max| ≫ 3 or std ≫ 1 is the policy pushing past what it saw in training.
print("\nNormalized action stats per dim (target: mean≈0, std≈1):")
for i in range(actions_norm.shape[1]):
    a = actions_norm[:, i]
    print(f"  dim {i}: mean={a.mean():+.3f}  std={a.std():.3f}  |max|={np.abs(a).max():.2f}")

# Per-dim histogram of executed (unnormalized) actions
n_dims = actions_unnorm.shape[1]
fig, axes = plt.subplots(1, n_dims, figsize=(2.4 * n_dims, 3))
for i, ax in enumerate(axes):
    ax.hist(actions_unnorm[:, i], bins=50)
    ax.set_title(f"action[{i}]"); ax.tick_params(labelsize=7)
plt.suptitle("Executed action distributions (unnormalized)", y=1.02)
plt.tight_layout(); plt.show()

### 11b. Deep-inspect a single episode

Runs one rollout with `--inspect_only` and dumps everything per step — state, normalized + unnormalized actions, and both camera frames — into a single `.npz`. Use this to diagnose a specific failure: load the file, scrub through frames, and check whether actions saturate, oscillate, or wander out of the joint limits.

In [ ]:
INSPECT_OUT = "inspect_episode.npz"

# Runs ONE rollout on libero_spatial / task 0 / seed 0 and dumps every step's
# state, action (norm + unnorm), and both camera frames. Cheap — under a minute.
!python -u lerobot-piper/src/benchmark_libero.py \
    --checkpoint {CHECKPOINT} \
    --suites libero_spatial \
    --inspect_only \
    --inspect_task_id 0 \
    --inspect_seed 0 \
    --inspect_output {INSPECT_OUT} \
    --image_size {IMAGE_SIZE} \
    --n_action_steps {N_ACTION_STEPS} \
    --max_steps {MAX_STEPS} \
    --headless \
    --skip_sanity_check

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ep = np.load(INSPECT_OUT, allow_pickle=True)
print(f"Suite: {ep['suite']}  Task: {ep['task_name']}  Success: {bool(ep['success'])}")
print(f"Description: {ep['task_description']}")
print(f"Steps: {len(ep['steps'])}, action_dim={ep['actions_unnorm'].shape[1]}")

# Per-dim action trajectory (unnormalized — i.e. what env.step actually saw)
actions = ep['actions_unnorm']
fig, ax = plt.subplots(figsize=(11, 4))
for i in range(actions.shape[1]):
    ax.plot(actions[:, i], label=f"a[{i}]")
ax.set_xlabel("step"); ax.set_ylabel("action (unnorm)")
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.set_title(f"Action trajectory — {ep['task_name']}  success={bool(ep['success'])}")
plt.tight_layout(); plt.show()

# Per-dim state trajectory
states = ep['states']
fig, ax = plt.subplots(figsize=(11, 4))
for i in range(states.shape[1]):
    ax.plot(states[:, i], label=f"s[{i}]")
ax.set_xlabel("step"); ax.set_ylabel("state (unnorm)")
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.set_title("Observation state trajectory")
plt.tight_layout(); plt.show()

# Sample 6 evenly-spaced agentview frames
agentview = ep['agentview']
T = len(agentview)
idxs = np.linspace(0, T - 1, 6, dtype=int)
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, i in zip(axes, idxs):
    ax.imshow(agentview[i]); ax.set_title(f"step {i}"); ax.axis("off")
plt.suptitle("agentview frames", y=1.02)
plt.tight_layout(); plt.show()

## 11. Inpect the data 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

import sys
import importlib # Import importlib
from pathlib import Path # Need to import Path

# Add lerobot-piper/src to sys.path if not already there
repo_src = Path("lerobot-piper/src").resolve()
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

if "benchmark_libero" in sys.modules: importlib.reload(sys.modules["benchmark_libero"]) # Use importlib.reload
from benchmark_libero import img_to_tensor # Corrected import statement


# ── Dataset frame 0 ─────────────────────────────────────────────────
ds = LeRobotDataset("lerobot/libero", revision="main")
sample = ds[0]
# LeRobot stores images as CHW float [0,1]; convert back to HWC uint8 for display
def to_hwc_uint8(t):
    arr = t.numpy() if hasattr(t, "numpy") else t
    if arr.ndim == 3 and arr.shape[0] in (1, 3):     # CHW → HWC
        arr = arr.transpose(1, 2, 0)
    if arr.dtype != np.uint8:
        arr = (arr * 255).clip(0, 255).astype(np.uint8)
    return arr

ds_main  = to_hwc_uint8(sample["observation.images.image"])
ds_wrist = to_hwc_uint8(sample["observation.images.image2"])

# ── Live env reset frame ────────────────────────────────────────────
task_suite = benchmark.get_benchmark_dict()["libero_spatial"]()
env = OffScreenRenderEnv(
    bddl_file_name=task_suite.get_task_bddl_file_path(0),
    camera_heights=256, camera_widths=256,
)
obs = env.reset()
env.close()

#live_main  = obs["agentview_image"]              # already HWC uint8
#live_wrist = obs["robot0_eye_in_hand_image"]

# Convert live obs through the FIXED img_to_tensor, then back to HWC for display
live_main_fixed  = (img_to_tensor(obs["agentview_image"]).permute(1,2,0).numpy() * 255).astype(np.uint8)
live_wrist_fixed = (img_to_tensor(obs["robot0_eye_in_hand_image"]).permute(1,2,0).numpy() * 255).astype(np.uint8)

# ── Plot 2×2 ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, img, title in zip(
    axes.flatten(),
    [ds_main, live_main_fixed, ds_wrist, live_wrist_fixed],
    ["Dataset — agentview", "Live env — agentview",
     "Dataset — wrist (eye-in-hand)", "Live env — wrist"],
):
    ax.imshow(img)
    ax.set_title(f"{title}\nshape={img.shape}  dtype={img.dtype}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

# Also print pixel-stat signatures so flips/inversions are unambiguous
for name, img in [("ds_main", ds_main), ("live_main", live_main_fixed),
                  ("ds_wrist", ds_wrist), ("live_wrist", live_wrist_fixed)]:
    print(f"{name:12s} mean={img.mean():6.1f}  top10rows.mean={img[:10].mean():6.1f}  bot10rows.mean={img[-10:].mean():6.1f}")